239. 滑动窗口最大值

给你一个整数数组 nums，有一个大小为 k 的滑动窗口从数组的最左侧移动到数组的最右侧。你只可以看到在滑动窗口内的 k 个数字。滑动窗口每次只向右移动一位。

返回 滑动窗口中的最大值 。 

示例 1：

输入：nums = [1,3,-1,-3,5,3,6,7], k = 3

输出：[3,3,5,5,6,7]

解释：

滑动窗口的位置                最大值

---------------               -----

[1  3  -1] -3  5  3  6  7       3

 1 [3  -1  -3] 5  3  6  7       3

 1  3 [-1  -3  5] 3  6  7       5

 1  3  -1 [-3  5  3] 6  7       5

 1  3  -1  -3 [5  3  6] 7       6

 1  3  -1  -3  5 [3  6  7]      7


示例 2：


输入：nums = [1], k = 1

输出：[1]
 

## 朴素的思想
我的思路很简单，维护最大值的索引，但是我发现如果最大值从左边窗口划过了，又要重新循环窗口算一遍最大值，这肯定不是最优雅的解法。而且提交后超过时间限制了，因此需要优化。

In [ ]:
class Solution:
    def maxSlidingWindow(self, nums: list[int], k: int) -> list[int]:
        if not nums or k == 0:
            return []
            
        ans = []
        # 记录滑动窗口最大值、以及其索引
        # 用 float('-inf') 比 -1e5 更严谨，因为测试用例里可能有极小的负数
        max_num, max_idx = float('-inf'), -1
        
        # 1. 初始化第一个窗口
        for i in range(k):
            # 重要：如果有多个相同的最大值，以最右边的那个为准 (保证它能在窗口里存活更久)
            if nums[i] >= max_num:
                max_num, max_idx = nums[i], i
        ans.append(max_num)
        
        # 2. 窗口开始向右滑动
        for i in range(k, len(nums)):
            # 情况 A: 新进来的数字极其强悍，直接刷新最大值
            if nums[i] >= max_num:
                max_num, max_idx = nums[i], i
            else:
                # 情况 B: 新进来的数字没那么强，这时候要看前任最大值还在不在窗口里
                # 当前窗口的范围是 [i - k + 1, i]，如果 max_idx < i - k + 1，说明它过期了
                if max_idx <= i - k:
                    # 这就是这个算法的“痛点”：最大值滑出去了，必须重新扫描当前整个窗口
                    max_num = float('-inf')
                    for j in range(i - k + 1, i + 1):
                        if nums[j] >= max_num:
                            max_num, max_idx = nums[j], j
                            
            # 把当前窗口的最大值加入结果
            ans.append(max_num)
            
        return ans

## 优化：单调队列

上面解法的最大问题在于：如果最大值从左边窗口划过了，又要重新循环窗口计算一遍最大值，如果每次都去重新扫描，时间复杂度会退化成O(N*K)，会超时。

关于 “最大值划出去了，谁来继任” 这个问题，算法界有一个极其优雅并且量身定制的数据结构：单调队列。

形象类比：现实生活中的**职场法则**。把滑动窗口想象成一个公司，窗口滑动就是时间的流逝，数字大小代表员工的能力。

单调队列的核心逻辑就是两条残酷的职场法则：

1. 如果一个新来的员工比你年轻，还比你强，那么你就要 say goodbye 了
2. 如果新来的员工没你强，但是他比你年轻，他可以先留下来当备胎

为什么叫单调队列？ 只要遵循上面的处理步骤，我们的队列里就会一直保持一个从大到小的单调递减顺序，队列的队首永远是当前窗口的最大值。

此外，我们还要考虑一种情况：老员工退休。也就是窗口右滑时，最左端的元素划出窗口外了。

下面给出代码实现：


In [ ]:
class Solution:
    def maxSlidingWindow(self, nums: List[int], k: int) -> List[int]:
        ans = []
        q = deque()
        for i, num in enumerate(nums):
            # 年轻力壮的员工踢走老弱病残员工
            while q and nums[q[-1]] <= num:
                q.pop()
            # 新员工牛马入职
            q.append(i)
            # 退休员工
            if q[0] <= i - k:
                q.popleft()
            # 记录窗口最大值
            if i >= k - 1:
                ans.append(nums[q[0]])
        return ans